# Identificacion de sentimientos a partir de videos.
A partir de un listado de videos guardados en google Drive.
### 1. Pasar de video a Fotos.

### Instalacion de librerias

In [ ]:
pip install deepface

### Importación de Librerias

In [ ]:
#de Videos a Imagenes.
import cv2
from google.colab import drive

#Extracción de emociones
import pandas as pd
from deepface import DeepFace

# Cantidad de imagenes generadas.
import os
import glob

# Monta la unidad de Google Drive
drive.mount('/content/drive')

### De Video a Imagenes, de Imagenes a Emociones ;)

In [ ]:
# Especifíca la ruta de la carpeta que contiene los videos
ruta_carpeta_videos = '/content/drive/MyDrive/0 New/Validacion DeepFace/DFEW Database/part_1/'  # Cambia esta ruta a la ruta de tu carpeta en Google Colab

# Usamos el módulo glob para buscar archivos con una extensión específica (en este caso, videos con extensión .mp4)
ruta_archivos_videos = os.path.join(ruta_carpeta_videos, '*.mp4')
lista_archivos_videos = glob.glob(ruta_archivos_videos)

# Contamos la cantidad de archivos encontrados  --- m es la cantidad de videos analizadas en el estudio.
m = len(lista_archivos_videos)  # Número de videos en la carpeta.

# Imprimimos el resultado
print(f'La cantidad de archivos de videos en la carpeta es: {m}')

# Creamos la base de datos que almacenará los videos y sus resultados.
df_BD = pd.DataFrame(columns=['Nombre de Imagen', 'angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy', 'Video', 'Q_images'])


# re-Asignamos Manualmente la cantidad de videos que queremos extraer. 
#m = 1000

# Loop sobre cada video: Extracción de frames del Video y Computación Afectiva de los frames.
for a in range (1, m+1):    # m es la cantidad de videos.-

    # Abrimos cada video en Google Drive, vamos cambiando el final del nombre del archivo que es un correlativo numérico.
    numero_video = a
    ruta_video = "/content/drive/MyDrive/0 New/Validacion DeepFace/DFEW Database/part_1/" + str(numero_video) + ".mp4"

    # Hacemos la función para extraer y almacenar las imagenes en la carpeta temporal del google Colab (sample data).
    def getFrame(sec):
        vidcap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
        hasFrames, image = vidcap.read()
        if hasFrames:
            cv2.imwrite("sample_data/image" + str(count) + ".jpg", image)     # Guardar el frame como archivo JPG
        return hasFrames

    # Abrir el video desde Google Drive
    vidcap = cv2.VideoCapture(ruta_video)

    # Obtener los FPS del video
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    print("FPS del video:", fps)

    # Calculamos de framerate en base a los fps del video.
    frameRate = 1 / fps


    #Hacemos la extraccion y almacenamiento de imagenes.
    sec = 0                     # Segundo desde donde empezamos a extraer
    count = 1
    success = True
    while success:
        sec = sec + frameRate
        sec = round(sec, 2)
        success = getFrame(sec)
        count = count + 1
    # Liberar el objeto VideoCapture y finalizar
    vidcap.release()
    cv2.destroyAllWindows()

    # COMPUTACION AFECTIVA de los frames del Video.
    # Especifica la ruta de la carpeta que contiene las imágenes
    ruta_carpeta = '/content/sample_data'  # Cambia esta ruta a la ruta de tu carpeta en Google Colab

    # Usamos el módulo glob para buscar archivos con una extensión específica (en este caso, imágenes con extensión .jpg)
    ruta_archivos = os.path.join(ruta_carpeta, '*.jpg')
    lista_archivos = glob.glob(ruta_archivos)

    # Contamos la cantidad de archivos encontrados  --- N es la cantidad de imagenes analizadas en el video.
    n = len(lista_archivos) #numero de imagenes en la carpeta.

    # Imprimimos el resultado
    print(f'La cantidad de archivos de imágenes en la carpeta es: {n}')

    # --------------------------------------------------------------------------------------------
    #EXTRACCION DE EMOCIONES A PARTIR DE IMAGENES.
    # Crear una lista para almacenar los datos
    data = []

    # Bucle para analizar cada imagen y guardar los datos en la lista (data)
    for i in range(1, n+1):     # n es la cantidad de archivos encontrados en la carpeta con formato de imagen (son los frames extraídos).
        # Analizamos la imagen con DeepFace
        img_path = f"sample_data/image{i}.jpg"  # Ruta de la imagen
        objs = DeepFace.analyze(img_path=img_path, actions=['emotion'], enforce_detection=False)
        
        # Obtener el nombre de la imagen y las emociones analizadas
        nombre_imagen = f"imagen{i}.jpg"
        emociones = objs[0]['emotion']
        
        # Agregar los datos a la lista
        data.append([nombre_imagen, emociones])
        
    # Crear un DataFrame de pandas a partir de la lista de datos
    df = pd.DataFrame(data, columns=['Nombre de Imagen', 'Emociones'])

    # Aplicar una función para convertir el diccionario de emociones en columnas individuales (una lista de varios elementos con valores a varias columnas).
    df = df.join(df['Emociones'].apply(pd.Series))

    # Eliminar la columna 'Emociones' original
    df = df.drop('Emociones', axis=1)

    # Redondear todos los valores en el DataFrame a un decimal.
    df = df.round(1)

    # Lista con el nuevo orden de las columnas.
    nuevo_orden = ['Nombre de Imagen', 'happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear' ] #'angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy']

    # Reordenamos las columnas en el DataFrame.
    df = df[nuevo_orden]

    # Agregamos el identificador del video.
    df=df.assign(Video=numero_video)

    # Agregamos columna que indica cantidad de imagenes extraidas del Video.
    df=df.assign(Q_images=n)

    # Agregamos dataframe de emociones del video (df) al dataframe que almacena los resultados de todos los videos (df2).
    df_BD = df_BD.append(df)

    # Eliminar archivos jpg de la carpeta. nombre carpeta. Con esto terminamos el loop.
    # Ruta a la carpeta que contiene las imágenes JPG
    ruta_carpeta = '/content/sample_data/'

    # Obtener la lista de archivos en la carpeta.
    archivos = os.listdir(ruta_carpeta)

    # Iterar sobre los archivos y eliminar las imágenes en formato JPG.
    for archivo in archivos:
        if archivo.endswith('.jpg'):
            ruta_archivo = os.path.join(ruta_carpeta, archivo)
            os.remove(ruta_archivo)

#Imprimimos el resultado
df_BD

### Guardamos la BD

In [ ]:
#Guardamos la Base de datos.
ruta_video

In [ ]:
import pandas as pd

# Guardamos el DataFrame df2.
# Ruta de destino en Google Colab
ruta_destino = '/content/sample_data/df_BD.csv'
# Guardar el DataFrame como archivo CSV
df_BD.to_csv(ruta_destino, index=False)
# Ruta de destino en Google Colab 
ruta_destino = '/content/sample_data/df_BD.xlsx'
# Guardar el DataFrame como archivo Excel
df_BD.to_excel(ruta_destino, index=False)

#copiamos la dataframe en df2.
df2 = df_BD.copy()

### Promedios de emocion por Video.

In [ ]:
### Promedios de emocion por Video.
# Columnas de emociones
columnas_emociones = ['happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear' ]
# Sumar los valores de las columnas de emociones para cada video y la divide por la cantidad de imagenes del video (valor de cualquier casilla de Q_imagenes del video)
df_mean = df2.groupby('Video')[columnas_emociones].mean().reset_index()
print(df_mean)

### Conteo  //  Contar Casos con prob mayor a un umbral. // 

In [ ]:
# Columnas de emociones
columnas_emociones = ['happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear' ]
# Umbral para filtrar los valores de las emociones
umbral = 60
# Aplicar la máscara booleana y contar las veces que se supera el umbral para cada video
df_cont_u60 = df2[columnas_emociones].applymap(lambda x: 1 if x > umbral else 0).groupby(df2['Video']).sum().reset_index()
# Imprimir el DataFrame de resultados
df_cont_u60

### Respaldamos la BD del Resumen de las observaciones de los videos en formato csv y xlsx.

In [ ]:
# Guardamos la BD en .csv y .xlsx .-
import pandas as pd

# Ruta de destino en carpeta temporal de Google Colab sample_data
ruta_destino = '/content/sample_data/nuevo_dataframedf_BD.csv'

# Guardar el DataFrame como archivo CSV
nuevo_dataframe.to_csv(ruta_destino, index=False)

# Ruta de destino en Google Colab
ruta_destino = '/content/sample_data/nuevo_dataframedf_BD.xlsx'

# Guardar el DataFrame como archivo Excel
nuevo_dataframe.to_excel(ruta_destino, index=False)

# EXTRAS // PAPELERA DE RECICLAJE.

### Eliminamos valores perdidos o NaNs

In [ ]:
# Eliminar los valores NaN en la columna 'Columna1'.
nuevo_dataframe_sin_nans = nuevo_dataframe2.dropna(subset=['happy_mean', 'sad_mean','happy_sum', 'neutral_mean', 'angry_mean', 'surprise_mean', 'disgust_mean', 'fear_mean', ])
# Mostrar el DataFrame sin los valores NaN.
nuevo_dataframe_sin_nans.head(5)